In [0]:
import hashlib
from pathlib import Path
from datetime import datetime, timezone
from pyspark.sql import functions as F

batch_id = "batch_001"

control_rows = (
    spark.table("workspace.migration_dev.batch_control")
    .filter(F.col("batch_id") == batch_id)
    .collect()
)

if len(control_rows) != 1:
    raise RuntimeError("Expected one batch control record")

control = control_rows[0]

if control["batch_status"] != "APPROVED" or control["failed_checks"] != 0:
    raise RuntimeError("Batch is not approved; downstream load blocked")

documents = (
    spark.table("workspace.migration_dev.gold_load_manifest")
    .filter(F.col("batch_id") == batch_id)
    .collect()
)

if len(documents) != control["expected_document_count"]:
    raise RuntimeError("Gold count differs from the approved batch")

# This driver-based loader is bounded for our small practice batches.
if len(documents) > 1000:
    raise RuntimeError("Practice loader supports at most 1,000 documents")

if any(
    row["validation_run_id"] != control["validation_run_id"]
    for row in documents
):
    raise RuntimeError("Gold approval is outdated")

for key in ["document_id", "file_name"]:
    if len({row[key] for row in documents}) != len(documents):
        raise RuntimeError(f"Duplicate {key}; load blocked")

source_folder = Path(
    f"/Volumes/workspace/migration_dev/landing/{batch_id}/pdfs"
)
destination_folder = Path(
    f"/Volumes/workspace/migration_dev/downstream/{batch_id}"
)
destination_folder.mkdir(parents=True, exist_ok=True)

acknowledgements = []
copied = 0
reused = 0

for row in documents:
    destination = None
    destination_hash = None
    loaded_at = None

    try:
        filename = row["file_name"]

        if not filename or Path(filename).name != filename:
            raise ValueError("Unsafe filename")

        source = source_folder / filename
        destination = destination_folder / filename

        if row["pdf_path"] != str(source):
            raise ValueError("Source path is outside the expected batch path")

        content = source.read_bytes()

        if hashlib.sha256(content).hexdigest() != row["source_sha256"]:
            raise ValueError("Source checksum mismatch; copy blocked")

        if destination.exists():
            reused += 1
        else:
            # Exclusive creation prevents overwriting an existing document.
            with destination.open("xb") as output:
                output.write(content)
            copied += 1

        destination_hash = hashlib.sha256(
            destination.read_bytes()
        ).hexdigest()

        if destination_hash != row["source_sha256"]:
            raise ValueError("Destination checksum mismatch")

        status = "SUCCESS"
        error = None
        loaded_at = datetime.now(timezone.utc)

    except Exception as exc:
        status = "FAILED"
        error = str(exc)

    acknowledgements.append((
        batch_id,
        row["document_id"],
        row["validation_run_id"],
        status,
        str(destination) if destination is not None else None,
        destination_hash,
        error,
        loaded_at,
    ))

ack_df = spark.createDataFrame(
    acknowledgements,
    """
    batch_id STRING,
    document_id STRING,
    validation_run_id STRING,
    load_status STRING,
    destination_path STRING,
    destination_sha256 STRING,
    error_message STRING,
    loaded_at TIMESTAMP
    """,
)

ack_df.createOrReplaceTempView("current_load_acknowledgements")

spark.sql("""
    MERGE INTO workspace.migration_dev.load_acknowledgements AS target
    USING current_load_acknowledgements AS source
    ON target.batch_id = source.batch_id
       AND target.document_id = source.document_id
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""")

print(f"New files copied: {copied}")
print(f"Existing files checked: {reused}")
display(ack_df.groupBy("load_status").count())

if any(row[3] == "FAILED" for row in acknowledgements):
    raise RuntimeError("Some loads failed; inspect load_acknowledgements")

New files copied: 0
Existing files checked: 100


load_status,count
SUCCESS,100
